In [14]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [15]:
#Datasets and Dataloaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms


#image = scale(0,1) = normalize =>(-1,1)
transform = transforms.Compose([
    transforms.ToTensor(), #convert image into pytorchtensor and scale image
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])


trainset = CIFAR10(root="./data",train = True,download = True,transform=transform)
testset = CIFAR10(root="./data",train=False,download=True,transform=transform)

In [16]:
trainloader = DataLoader(trainset,batch_size = 64,shuffle = True)
testloader = DataLoader(testset,batch_size = 64)

# Build CNN

In [17]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.Conv_layers = nn.Sequential(
            nn.Conv2d(3, 32,kernel_size=3,padding=1), #i/p , o/p
            nn.ReLU(),
            nn.MaxPool2d(2,2), #kernelsize=2 and stride = 2

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2) ,

            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )


        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),

            nn.Linear(256,10)
        )


    def forward(self,x):
        x = self.Conv_layers(x)
        x = x.view(x.size(0),-1) #flattening
        x = self.fc_layers(x)

        return x

In [18]:
model = CNN()

In [19]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Training CNN

In [ ]:
epochs = 10
for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images,labels in trainloader:
        optimizer.zero_grad()
        output = model.forward(images) #FP
        loss = criterion(output,labels) #loss fnx
        loss.backward() #BP

        optimizer.step() #update params

        epoch_training_loss += loss.item()

        #validation
    model.eval()
    running_eval_loss = 0.0

        with torch.no_grad():
            for images,labels in testloader:
                output = model(images)
                loss = criterion(output,labels)
                running_eval_loss += loss.item()
                
    epoch_val_loss = running_eval_loss / len(testloader)
    print(f"epoch{epoch+1}/{epochs} &loss={epoch_training_loss/len(trainloader)} & validation loss {epoch_val_loss}")

In [ ]:
#Evaluate our CNN
correct_labels = 0
total_labels = 0

model.eval()
with torch.no_grad(): #no backward
    for images,labels in testloader:
        outputs = model.forward(images)
        _,predicted = torch.max(outputs,1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy{correct_labels / total_labels*100}")